# Phase 3: TF-IDF + Logistic Regression Baseline

This notebook trains a simple text classification baseline using the cleaned ticket descriptions from Phase 2. It predicts `Issue_Category`. It does not use transformers, embeddings, or retrieval.

We use only the `text` column, which contains `Ticket_Description` after minimal whitespace cleaning. `Ticket_Subject` remains excluded because Phase 1 showed a strong association with the category that could create a shortcut.

## 1. Load the cleaned dataset

**Why:** Starting from Phase 2's saved file keeps this baseline consistent with the inspected and cleaned data.

In [ ]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/processed/cleaned_tickets.csv")
tickets = pd.read_csv(data_path)

print(f"Rows: {len(tickets)}")
tickets.head()

## 2. Recreate the train/test split

**Why:** We use the same 80/20 stratified split and fixed random seed as Phase 2, so the evaluation is reproducible and the category proportions are similar in both sets. `X` is the ticket text; `y` is the category we want to predict.

In [ ]:
from sklearn.model_selection import train_test_split

X = tickets["text"]
y = tickets["Issue_Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 3. Turn text into TF-IDF features

**Why:** A classifier needs numbers as input. TF-IDF represents a ticket using word and two-word phrase features, giving more weight to terms that are useful in a document and less weight to terms common across many documents.

We fit the vectorizer using **only `X_train`**. If we fit it on all tickets, the vocabulary and document frequencies would use information from the test set. That is data leakage: the test data would influence the representation before evaluation, making the test score less independent. We then use the already-fitted vectorizer to transform both sets.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Original X_train shape:", X_train.shape)
print("TF-IDF training matrix shape:", X_train_tfidf.shape)
print("Learned vocabulary features:", len(vectorizer.get_feature_names_out()))
print("Example feature names:", vectorizer.get_feature_names_out()[:20])

The TF-IDF matrix has one row per ticket and one column per learned feature. The current configuration allows up to 10,000 features; the actual count can be lower.

## 4. Train Logistic Regression

**Why:** Logistic Regression is a straightforward classifier that works well with sparse text features and provides a useful baseline. `max_iter=1000` gives the solver enough iterations to converge; we are not tuning model settings.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print("Model trained.")

## 5. Evaluate on the held-out test set

**Why:** The test set contains tickets the model did not use for fitting. Its scores estimate how well this baseline handles unseen examples from the same dataset.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted", zero_division=0
)

print(f"Accuracy:          {accuracy:.3f}")
print(f"Macro precision:   {macro_precision:.3f}")
print(f"Macro recall:      {macro_recall:.3f}")
print(f"Macro F1:          {macro_f1:.3f}")
print(f"Weighted precision:{weighted_precision:.3f}")
print(f"Weighted recall:   {weighted_recall:.3f}")
print(f"Weighted F1:       {weighted_f1:.3f}")

### What these metrics mean

- **Accuracy** is the fraction of all tickets whose predicted category is correct.
- **Precision** asks: of the tickets predicted as a category, what fraction truly belong to it?
- **Recall** asks: of the tickets truly in a category, what fraction did the model find?
- **F1** combines precision and recall using their harmonic mean, so it is high only when both are high.
- **Macro average** calculates a metric for each category and gives every category equal weight. This is useful when a smaller group such as Fraud matters too.
- **Weighted average** also calculates each category separately, then weights results by how many examples that category has. Larger categories influence it more.

Accuracy can hide poor results on a small category. We therefore also inspect class-level scores and the confusion matrix.

## 6. Per-category report

**Why:** A classification report shows precision, recall, F1, and support (number of test examples) for each category, helping us see whether any category is being missed.

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

On this split, all five categories have the same perfect scores, so there is no single best or worst category by the measured metrics. This result is unusually strong for real-world support tickets; check the confusion matrix and examples below, and treat the score as a property of this dataset and split rather than a guarantee for future tickets.

## 7. Confusion matrix

**Why:** The confusion matrix counts actual categories against predicted categories. Off-diagonal values show which categories the model confuses.

In [ ]:
import matplotlib.pyplot as plt

class_names = model.classes_
matrix = confusion_matrix(y_test, y_pred, labels=class_names)

plt.figure(figsize=(7, 6))
plt.imshow(matrix, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted category")
plt.ylabel("Actual category")
plt.xticks(range(len(class_names)), class_names, rotation=35, ha="right")
plt.yticks(range(len(class_names)), class_names)

for row in range(len(class_names)):
    for column in range(len(class_names)):
        plt.text(column, row, matrix[row, column], ha="center", va="center")

plt.colorbar(label="Number of tickets")
plt.tight_layout()
plt.show()

off_diagonal_count = matrix.sum() - matrix.trace()
print(f"Misclassified test tickets: {off_diagonal_count}")

There are no off-diagonal counts for this split, so no categories are commonly confused in these test predictions. In a less perfect result, the largest off-diagonal cells would point to the category pairs to investigate.

## 8. Inspect prediction examples

**Why:** Reading individual predictions helps us connect numerical scores to actual ticket text and notice possible data patterns that aggregate metrics do not show. We show incorrect examples first when available, followed by correct examples.

In [ ]:
prediction_examples = pd.DataFrame({
    "ticket_text": X_test,
    "actual_category": y_test,
    "predicted_category": pd.Series(y_pred, index=X_test.index),
})
prediction_examples["correct"] = (
    prediction_examples["actual_category"] == prediction_examples["predicted_category"]
)

incorrect_examples = prediction_examples[~prediction_examples["correct"]].head(10)
correct_examples = prediction_examples[prediction_examples["correct"]].head(
    10 - len(incorrect_examples)
)
examples_to_show = pd.concat([incorrect_examples, correct_examples])
examples_to_show

This dataset produces no incorrect predictions on the current test split, so the table contains ten correct examples. That makes it especially important to check whether the dataset is representative and whether its text contains repeated or category-revealing patterns. A future evaluation on independently collected tickets would be a stronger test of real-world performance.

## 9. Save the trained artifacts

**Why:** Saving the fitted vectorizer and classifier lets us reuse them later without refitting. The vectorizer and model must be used together: new text must first be transformed with this same vectorizer.

In [ ]:
import joblib

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

vectorizer_path = models_dir / "tfidf_vectorizer.pkl"
model_path = models_dir / "logistic_regression.pkl"

joblib.dump(vectorizer, vectorizer_path)
joblib.dump(model, model_path)

print(f"Saved vectorizer to {vectorizer_path}")
print(f"Saved classifier to {model_path}")

## 10. Try one new ticket

**Why:** This small example shows how the two saved components work together for one incoming ticket. It does not create a web app.

In [ ]:
new_ticket = "I was charged twice for my subscription"

new_ticket_tfidf = vectorizer.transform([new_ticket])
predicted_category = model.predict(new_ticket_tfidf)[0]

print("Ticket:", new_ticket)
print("Predicted category:", predicted_category)

## Baseline Results

- **Model:** TF-IDF with Logistic Regression (`max_iter=1000`).
- **Features:** Unigrams and bigrams from cleaned `Ticket_Description`; 7,581 learned features with the current data and settings.
- **Training size:** 16,000 tickets.
- **Test size:** 4,000 tickets.
- **Accuracy:** 1.000.
- **Macro F1:** 1.000.
- **Weighted F1:** 1.000.

### Main weaknesses to keep in mind

The perfect test score and zero errors are unusually strong. This may mean the dataset is synthetic or contains very clear/repeated category wording. The model may not generalize to messy real support requests, new issue types, or different writing styles. `Ticket_Subject` remains excluded because of its category association. Before presenting this as real-world performance, inspect the dataset's origin and evaluate on new, representative tickets.